In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

### AEUの時系列データ取得

In [3]:
code = 'AEU'
market = 'AX'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [4]:
timeseries_df = request_api.get_stock_time_series_data(
    code=code,
    market=market,
    start=start,
    end=end
)
timeseries_df

取得件数: 5429


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1191758,AEU,AX,2004-11-19,-14.489446,-14.489446,-14.489446,-14.489446,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1191759,AEU,AX,2004-11-22,-18.629290,-20.492218,-17.801321,-18.629290,1216,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1191760,AEU,AX,2004-11-23,-22.769131,-24.011084,-19.664249,-19.664249,803,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1191761,AEU,AX,2004-11-24,-23.183113,-24.011081,-22.355145,-23.597096,346,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1191762,AEU,AX,2004-11-25,-22.355145,-23.183113,-21.941160,-23.183113,123,-19.912639,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5424,1197182,AEU,AX,2026-05-04,0.400000,0.400000,0.370000,0.395000,683860,0.392000,...,0.444528,0.380272,False,NaN,NaN,NaN,NaN,NaN,NaN,False
5425,1197183,AEU,AX,2026-05-05,0.375000,0.390000,0.355000,0.375000,436581,0.385000,...,0.444115,0.382285,False,NaN,NaN,NaN,NaN,NaN,NaN,False
5426,1197184,AEU,AX,2026-05-06,0.380000,0.390000,0.370000,0.370000,77419,0.380000,...,0.444115,0.382285,False,NaN,NaN,NaN,NaN,NaN,NaN,False
5427,1197185,AEU,AX,2026-05-07,0.390000,0.390000,0.355000,0.370000,982174,0.376000,...,0.442942,0.385858,False,NaN,NaN,NaN,NaN,-58.333333,NaN,False


### Sprott Physical Uranium Trust Fundの時系列データを取得

Yahoo financeはウランスポット価格に関する時系列データを提供していないためSprottのETFで代替

In [11]:
uran = 'U-UN'
uran_market_code = 'TO'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=uran,
    market=uran_market_code,
    start=start,
    end=end
)
response

{'result': True}

In [12]:
uran_timeseries_df = request_api.get_stock_time_series_data(
    code=uran,
    market=uran_market_code,
    start=start,
    end=end
)
uran_timeseries_df

取得件数: 5020


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1197187,U-UN,TO,2006-05-09,9.050000,9.140000,8.600000,9.140000,518800,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1197188,U-UN,TO,2006-05-10,9.400000,9.470000,9.060000,9.060000,493600,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1197189,U-UN,TO,2006-05-11,9.140000,9.500000,9.110000,9.500000,1065200,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1197190,U-UN,TO,2006-05-12,8.850000,9.140000,8.710000,9.140000,460300,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1197191,U-UN,TO,2006-05-15,8.400000,9.000000,8.380000,8.850000,519800,9.138,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5015,1202202,U-UN,TO,2026-05-04,27.610001,28.110001,27.250000,27.690001,608500,28.122,...,28.521374,27.646626,True,NaN,NaN,NaN,NaN,NaN,NaN,False
5016,1202203,U-UN,TO,2026-05-05,27.700001,28.110001,27.219999,27.850000,638500,28.090,...,28.496009,27.743191,False,NaN,28.1196,NaN,NaN,NaN,NaN,False
5017,1202204,U-UN,TO,2026-05-06,27.940001,28.200001,27.600000,27.950001,668000,28.016,...,28.484538,27.807462,False,NaN,NaN,NaN,NaN,NaN,NaN,False
5018,1202205,U-UN,TO,2026-05-07,27.510000,28.180000,27.290001,27.940001,1331400,27.984,...,28.480452,27.798749,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [15]:
def stock_prices_and_material_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        market: str | None =  None,
        df_sp: pd.DataFrame | None = None,
        df_mat1: pd.DataFrame | None = None,
        df_mat2: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=market,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # mat1価格を統合
    if df_mat1 is not None:
        df_mat1_tmp = df_mat1.copy() if df_mat1 is not None else pd.DataFrame()
        if "date" not in df_mat1_tmp.columns:
            df_mat1_tmp = df_mat1_tmp.reset_index()
        df_mat1_tmp["date"] = pd.to_datetime(df_mat1_tmp["date"])
        df_mat1_tmp = df_mat1_tmp.set_index("date")
        df_mat1_tmp = df_mat1_tmp.loc[start:end]

    # mat2価格を統合
    if df_mat2 is not None:
        df_mat2_tmp = df_mat2.copy() if df_mat2 is not None else pd.DataFrame()
        if "date" not in df_mat2_tmp.columns:
            df_mat2_tmp = df_mat2_tmp.reset_index()
        df_mat2_tmp["date"] = pd.to_datetime(df_mat2_tmp["date"])
        df_mat2_tmp = df_mat2_tmp.set_index("date")
        df_mat2_tmp = df_mat2_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_mat1 is not None:
        df["MA5_MAT1"] = df_mat1_tmp["ma5"].reindex(df.index)
        df["MA25_MAT1"] = df_mat1_tmp["ma25"].reindex(df.index)
    if df_mat2 is not None:
        df["MA5_MAT2"] = df_mat2_tmp["ma5"].reindex(df.index)
        df["MA25_MAT2"] = df_mat2_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT1（右軸） ---
    if df_mat1 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT1"],
                name="MAT1_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT1"],
                name="MAT1_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT2（左軸） ---
    if df_mat2 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT2"],
                name="MAT2_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT2"],
                name="MAT2_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [16]:
name = "Atomic Eagle Ltd"
start = dt.datetime(2025, 12, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_material_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    market=None,
    df_sp=None,
    df_mat1=None,
    df_mat2=None
)
fig.show()

取得件数: 304


In [18]:
name = "Sprott Physical Uranium Trust Fund"
start = dt.datetime(2025, 12, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_material_prices(
    code=uran,
    name=name,
    start=start,
    end=end,
    market=uran_market_code,
    df_sp=None,
    df_mat1=None,
    df_mat2=None
)
fig.show()

取得件数: 302


In [9]:
response = request_api.update_corp_finance_data(
    code=code,
    market=market
)
response

{'result': True}

In [10]:
aeu_financials_data = request_api.get_corp_financials_data(code=code, market=market)
aeu_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=code, market=market)
aeu_cash_flow_data = request_api.get_corp_cash_flow_data(code=code, market=market)
aeu_earnings_data = request_api.get_corp_earnings_data(code=code, market=market)
aeu_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=code, market=market)

GET response: {'detail': 'Quarterly earnings data not found'}


In [19]:
# Muntanga Uranium Project NI 43-101 Technical Report
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://minedocs.com/28/Muntanga-FS-01233025.pdf",
    directory_path="/workspace/data",
)
md_file_path

markitdown failed (rc=-9). Falling back to pdftotext if available. stderr:
onnxruntime cpuid_info warning: Unknown CPU vendor. cpuinfo_vendor value: 0
Generated (pdftotext fallback): /workspace/data/Muntanga-FS-01233025.pdf.md


'/workspace/data/Muntanga-FS-01233025.pdf.md'